Try to Implement FireGnn on HeteroGraph

In [144]:
import pickle
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, GINConv
from torch_geometric.data import Data
from torch_geometric.nn import HeteroConv, GATConv, HGTConv
from torch_geometric.data import HeteroData
import numpy as np
import networkx as nx

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from collections import Counter
import scipy.stats
from tqdm import tqdm
import pandas as pd
import re
from typing import Any, Dict, Tuple



### Design
+ Graph: Map patients as nodes onto AD-KG: edges are created between patient and proteins if the protein exist in gene-expression data;
    + patient_node.y = labels;
    + patient_protein_edge_weight = gene expression values;
    + all the other nodes: node.y = (association_score_to_AD > 0.5)?
    + 
+ HeteroData:
    + data.x:
    + data.y:
    + data.relevance:
    + data.topo_features:

#### (a) using conditional rules to give various emphasis on patient-protein edge weight
+ ConditioanlMessageLayer:
    + Conditional message passing:
    + patient - protein edge:
      + $ r_1(v) $ = relevance score of non-patient node;
      + $ r_2(u) = \begin{cases} 1 & \text{if }  y_u = 1 \\ 0 & \text{if }  y_u = 0 \end{cases}$, where $u$ is patient ndoe;
      + $ r_3(u,v) = $e_{u,v}, the edge_weight between node $u$ and $v$;
      + combined gating rule:
        + $ g_{u,v} = \sigma(\alpha (r_1(v)*r_3(u,v) - \theta)) * r_2(u) $; 
        + $ g_{u,v} \in [0,1]$
    + all other edges gating:
      + $g_{u,v}$ = edge weight
        + edge_weight = embedding similarity between nodes;(`current implementation`)
        + edge_weight = the average node_relevance of two nodes;(`prefer`)
        + or edge_weight = 1: `not good because gated_edge_weight of edge_patient&protein < 1`
  
+ GNNModel
  + instead of combining gate-layer and GNN layer before the final classifier layer, here the rule-layer should be incorporated in each GNN layer;
  + incorporate gated_edge_weight to GAT layer.

#### (b) Message Passing Gating to each GAT layer
+ ConditionalMessageLayer
+ simple_GATConv combine the output of ConditioanlMessageLayer
+ HeteroGATModel to wrap all simple_GATConv for all edge_types
#### (c) Conditional Attention coefficient
+ still need to explore

## Prepare Patient-Graph

load data

In [45]:
ad_kg_path = "./data/KG/ad_network_with_reverse_edges.pkl"
adni_exp_path = "./data/adni_gene_cleaned.csv"
adni_target_path = "./data/adni_targets.tsv"

# load graph
with open(ad_kg_path, 'rb') as f:
    kg = pickle.load(f)
# read patient gene expression data
exp = pd.read_csv(adni_exp_path, index_col=0)
exp = exp.transpose()

# read patient labels
design = pd.read_csv(adni_target_path, sep='\t', index_col = 0)
design['Target'] = design['Target'].map({'Control':0, 'Disease':1})
labels = design['Target'].to_list()

map patients onto KG

In [ ]:
def extract_hgnc(node_id):
    match = re.search(r'HGNC:"([^"]+)"\)', node_id)
    return match.group(1) if match else ""

def add_patient_to_kg(kg:nx.MultiDiGraph, exp:pd.DataFrame, output_filename:str):
    """Add patients to KG with gene-expression-value as edge_weight, 
    create edges between patients and all overlapping HGNC-proteins in KG and expression data .

    Args:
        kg (nx.MultiDiGraph): knowledge graph
        exp (pd.DataFrame): patient gene expression data (num_patients x num_genes)
        output_filename(str): save graph

    Returns:
        nx.MultiDiGraph: A new Knowledge Graph with patient nodes
    """
    exp_genes = list(exp.columns)
    print('The number of genes in Gene Expression data:',len(exp_genes))
    patients = list(exp.index)
    print('The number of patients:', len(patients))

    # add patients to kg
    G = kg.copy()
    kg_proteins = [node for node in G.nodes(data=True) if node[1]['label'] == 'Protein']
    print('The number of KG proteins: ',len(kg_proteins))

    mapped_nodes = set()
    for i in tqdm(range(len(patients)), desc='Add patients to KG'):
        head = patients[i]
        target = labels[i]
        if head not in G.nodes:
            G.add_node(head, label='Patient', y=target)

        for node,attr in kg_proteins:
            
            gene_name = extract_hgnc(node) # only link to HGNC proteins

            if gene_name in exp_genes:
                j = exp_genes.index(gene_name)
                #print(head)
                #print(gene_name)
                exp_value = exp.iloc[i,j]
                #print(exp_value)
                G.add_edge(head, node, 'express', edge_weight = exp_value)
                G.add_edge(node, head, 'rev_express', edge_weight = exp_value)
                mapped_nodes.add(node)
    print(f'The number of mapped protein-nodes is {len(mapped_nodes)}')
    
    # save graph
    with open(output_filename, 'wb') as f:
        pickle.dump(G, f)
    print('-------------------------- Done ------------------------------')
    return G, mapped_nodes

In [43]:
G, mapped_nodes = add_patient_to_kg(kg, exp, './data/KG/patient_kg.pkl')

The number of genes in Gene Expression data: 19100
The number of patients: 744
The number of KG proteins:  1301


Add patients to KG: 100%|██████████| 744/744 [05:01<00:00,  2.46it/s]


The number of mapped protein-nodes is 735
-------------------------- Done ------------------------------


## Prepare HeteroData

(1) Get a AD relevance score for each node
+ cosine similarity / biliner similarity between each node and AD-node;
  + cosine similarity: $ cos(X,y) = \frac{X \cdot y}{||X|| \cdot ||y||} $
  + bilinear similarity: add a light learnable weight matrix to scale cosine similarity, such that it can learn the similarity in a AD-disease-aware way. $ bil(X,y) = X^T \cdot W \cdot y + b $
   
+ graph-aware relavence propagation
  + the similarity scores only capture node-level relevant, a following propagation is better to be applied to get a netwrok-level relevance --> relevant nodes' neighborhood are also relevant;
  + can train a edge-type-aware relevance model, but i don't think necessary.

a. cosine similarity

In [ ]:
with open('./data/hgt_kge_node_mappings.pkl', 'rb') as f:
    node_mappings = pickle.load(f)
kg_embed = torch.load('./data/hgt_KGEmbeddings.pt')


# compute cosine similarity to ad_embed
def get_cosine_similarity(kg_embed:dict, node_mappings:dict, output_type:str='list'):
    """calculate cosine similarity between all nodes in KG and AD-node.

    Args:
        kg_embed (dict): knowledge graph embeddings
        node_mappings (dict): node mappings
        output_type (str): choose between [dict, list]

    Returns:
        dict: contains cosine similarity scores
    """
    ad_idx = node_mappings['Pathology']['path(MESH:"Alzheimer Disease")']
    ad_embed = kg_embed['Pathology'][ad_idx]

    node_relevances_list = {}
    node_relevances_dict = {}
    for node_type, node_info in node_mappings.items():
        node_relevances_dict[node_type] = {}
        node_relevances_list[node_type] = []
        for node_name, node_idx in node_info.items():
            node_embed = kg_embed[node_type][node_idx]
            score = F.cosine_similarity(node_embed, ad_embed, dim=0)
            node_relevances_dict[node_type][node_name] = score
            node_relevances_list[node_type].append(score)
    
    if output_type == 'dict':
        return node_relevances_dict
    else:
        return node_relevances_list

node_relevances = get_cosine_similarity(kg_embed, node_mappings)

shortest path to AD as relevance score

In [ ]:
def get_shortest_path(G:nx.MultiDiGraph):
    pass

b. bilinear similarity

In [ ]:
# already have shgt_embeddings, can use cosine similarity as relevance score
class BilinearSimilarity(nn.Module):
    def __init__(self, embedding_1_dim, embedding_2_dim):
        super().__init__()
        self.W = nn.Parameter(torch.randn(embedding_1_dim, embedding_2_dim))
        self.bias = nn.Parameter(torch.Tensor(1))

    def forward(self, node_emb, ad_emb):
        """
        node_emb: [N, d]
        ad_emb:   [d] or [1, d]
        """
        if ad_emb.dim() == 1:
            ad_emb = ad_emb.unsqueeze(0)  # [1, d]

        # h_v^T W h_AD
        scores = torch.matmul(node_emb, self.W) @ ad_emb.T  # [N, 1]
        scores = scores.squeeze(-1)

        return torch.sigmoid(scores)  # relevance in (0,1)

class BiLinearSimilarity(nn.Module):
 
    def __init__(self, tensor_1_dim, tensor_2_dim, activation=None):
        super(BiLinearSimilarity, self).__init__()
        self.weight_matrix = nn.Parameter(torch.Tensor(tensor_1_dim, tensor_2_dim))
        self.bias = nn.Parameter(torch.Tensor(1))
        self.activation = activation
        self.reset_parameters()
 
    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight_matrix)
        self.bias.data.fill_(0)
 
    def forward(self, tensor_1, tensor_2):
        intermediate = torch.matmul(tensor_1, self.weight_matrix)
        result = (intermediate * tensor_2).sum(dim=-1) + self.bias
        if self.activation is not None:
            result = self.activation(result)
        return result

# need to mannualy assign labels to nodes in KG 
# (based on edge-distance to AD? ---- I think i will take this one
# or based on prior knowledge? ) -> at most combine this one as well.
def train_bilinear(
    model,
    node_emb,
    ad_emb,
    labels,
    optimizer,
    epochs=200
):
    """
    labels: [N] ∈ {0,1}
    """
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        scores = model(node_emb, ad_emb)
        loss = F.binary_cross_entropy(scores, labels.float())

        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"Epoch {epoch:03d} | Loss {loss.item():.4f}")


c. graph-aware relavence training

In [ ]:
def relevance_propagation(
    edge_index,
    init_relevance,
    num_nodes,
    alpha=0.85,
    num_iters=20
):
    """
    edge_index: [2, E]
    init_relevance: [N] in (0,1)
    """
    r0 = init_relevance
    r = r0.clone()

    # Build degree-normalized adjacency
    row, col = edge_index
    deg = torch.bincount(row, minlength=num_nodes).float()
    deg_inv = 1.0 / (deg + 1e-6)

    for _ in range(num_iters):
        r_new = torch.zeros_like(r)

        # message passing: r_j -> r_i
        r_new.index_add_(
            0,
            row,
            r[col] * deg_inv[col]
        )

        r = alpha * r_new + (1 - alpha) * r0

    return r


(2) add edge_weight to graph as cosine similarity between embeddings

In [67]:
for e in G.edges(data=True, keys=True):
    print(e)
    break

('g(HGNC:"BDNF")', 'p(HGNC:"APP",frag("672_713"))', 'decreases', {'annotation': {'confidence': "['Medium']", 'subgraph': "['Non-amyloidogenic subgraph', 'MAPK-ERK subgraph']"}, 'disease': 'ad', 'evidence': '["Using screening approaches in primary neurons, we identified brain- derived neurotrophic factor (BDNF) as a major inducer of Sorla that activates receptor gene transcription through the ERK (extracellular regulated kinase) pathway.These findings demonstrate that the beneficial effects ascribed to BDNF in APP metabolism act through induction of Sorla that encodes a negative regulator of neuronal APP processing"]', 'pmid': '[20007471]'})


In [76]:
for source, target, rel_type, edge_attr in G.edges(data=True, keys=True):
    src_type = G.nodes[source]["label"]
    dst_type = G.nodes[target]["label"]
    if src_type == 'Patient' or dst_type == 'Patient':
        # get weight
        weight = edge_attr['edge_weight']
        print(weight)
        break
        
    

2.6365


In [ ]:
def get_edge_weights(G:nx.MultiDiGraph, kg_embed:Dict, node_mappings:Dict, type:str):
    """edge_weight between nodes: 
       - patient ~ protein: gene expression value
       - other edges: (1) cosine similarity between embeddings; 
                      (2) average of nodes' relevance score;
                      
    Args:
        G (nx.MultiDiGraph): graph with patient nodes
        kg_embed (dict): embeddings of knowledge graph nodes
        node_mappings (dict): _description_
        type (str): 'cosine similarity', 'average relevance'

    Returns:
        Dict[Tuple[str, str, str], list]: edge weights
    """
    edge_weight_list:Dict[Tuple[str, str, str], list] = {}
    for source, target, rel_type, edge_attr in G.edges(data=True, keys=True):
        
        src_type = G.nodes[source]["label"]
        dst_type = G.nodes[target]["label"]
        if src_type != 'Patient' and dst_type != 'Patient':
            # get source and target embeddings
            src_idx = node_mappings[src_type][source]
            dst_idx = node_mappings[dst_type][target]
            
            src_embed = kg_embed[src_type][src_idx]
            dst_embed = kg_embed[dst_type][dst_idx]

            # calculate cosine similarity
            score = F.cosine_similarity(src_embed, dst_embed, dim=0)
            edge_type_tuple = (src_type, str(rel_type), dst_type)
            if edge_type_tuple not in edge_weight_list:
                edge_weight_list[edge_type_tuple] = []
        
            edge_weight_list[edge_type_tuple].append(score)
        
        if src_type == 'Patient' or dst_type == 'Patient':
            # get weight
            weight = torch.tensor(edge_attr['edge_weight'])
            edge_type_tuple = (src_type, str(rel_type), dst_type)
            if edge_type_tuple not in edge_weight_list:
                edge_weight_list[edge_type_tuple] = []
            #print(edge_type_tuple)
            edge_weight_list[edge_type_tuple].append(weight)
        
    return edge_weight_list

In [ ]:
edge_weight_list = get_edge_weights(G, kg_embed, node_mappings)


{('Gene', 'decreases', 'Protein'): [tensor(-0.1289),
  tensor(-0.0594),
  tensor(-0.3320),
  tensor(-0.3143),
  tensor(-0.0679)],
 ('Gene', 'increases', 'Gene'): [tensor(0.7472)],
 ('Gene', 'increases', 'Protein'): [tensor(0.0380),
  tensor(0.0761),
  tensor(0.0147),
  tensor(-0.1528),
  tensor(0.2237),
  tensor(-0.0565),
  tensor(-0.0245),
  tensor(-0.1737),
  tensor(-0.1968),
  tensor(-0.0662),
  tensor(-0.1556),
  tensor(-0.0010),
  tensor(-0.0565),
  tensor(-0.0313),
  tensor(-0.0373),
  tensor(0.1717),
  tensor(-0.0555),
  tensor(-0.0530),
  tensor(-0.0175),
  tensor(-0.0461),
  tensor(-0.0193),
  tensor(-0.1249),
  tensor(-0.0479),
  tensor(-0.0868),
  tensor(-0.0868),
  tensor(-0.1550)],
 ('Gene', 'rev_association', 'Gene'): [tensor(0.8187), tensor(0.8644)],
 ('Gene', 'rev_association', 'Protein'): [tensor(0.1693),
  tensor(-0.1044),
  tensor(0.0435),
  tensor(0.0435),
  tensor(0.0844),
  tensor(0.1088),
  tensor(-0.0498),
  tensor(0.0349),
  tensor(0.0324),
  tensor(-0.0097),
 

In [94]:
w1 = edge_weight_list[('Gene', 'decreases', 'Protein')]
torch.stack(w1).float()

tensor([-0.1289, -0.0594, -0.3320, -0.3143, -0.0679])

(3) convert nx.Multigraph to HeteroData
+ data.x
+ data.y
+ data.edge_index
+ data.edge_weight: 
+ data.relevance
+ data.train_mask
+ data.val_mask
+ data.test_mask
+ data.topo_features (optional)

In [95]:
def networkx_to_hetero_data(graph: nx.MultiDiGraph, node_relevances, y_labels, edge_weight_list) -> Tuple[HeteroData, Dict[str, Dict[Any, int]]]:
    """Convert a NetworkX heterogeneous graph to HeteroData."""
    data = HeteroData()
    node_mappings: Dict[str, Dict[Any, int]] = {}

    for node_id, attrs in graph.nodes(data=True):
        node_type = attrs.get("label")
        if node_type not in node_mappings:
            node_mappings[node_type] = {}
        if node_id not in node_mappings[node_type]:
            node_mappings[node_type][node_id] = len(node_mappings[node_type])
    
    # add data attributes
    #data.relevance
    for node_type, mapping in node_mappings.items():
        data[node_type].num_nodes = len(mapping)
        if node_type != 'Patient':
            data[node_type].relevance = node_relevances[node_type]
    # data.y
    data['Patient'].y = y_labels
    # data.x
    data.x = torch.nn.ModuleDict({
            node_type: torch.nn.Embedding(num_nodes, 64)
            for node_type, num_nodes in {nt: data[nt].num_nodes for nt in data.node_types}.items()
        })

    print(f"Found {len(node_mappings)} node types.")

    # edge_index
    edge_lists: Dict[Tuple[str, str, str], list] = {}
    for source, target, rel_type in graph.edges(keys=True):
        src_type = graph.nodes[source]["label"]
        dst_type = graph.nodes[target]["label"]
        edge_type_tuple = (src_type, str(rel_type), dst_type)
        if edge_type_tuple not in edge_lists:
            edge_lists[edge_type_tuple] = []
        edge_lists[edge_type_tuple].append(
            [
                node_mappings[src_type][source],
                node_mappings[dst_type][target],
            ]
        )

    for edge_type_tuple, edges in edge_lists.items():
        data[edge_type_tuple].edge_index = (
            torch.tensor(edges, dtype=torch.long).t().contiguous()
        )
    print(f"Found {len(edge_lists)} edge types.")

    # edge_weight
    for edge_type_tuple, edge_weights in edge_weight_list.items():
        data[edge_type_tuple].edge_weight = torch.stack(edge_weights).float()
    print("Add edge_weights to HeteroData")

    
    print("Conversion complete!")
    return data, node_mappings

In [98]:
data, new_node_mappings = networkx_to_hetero_data(G, node_relevances, labels, edge_weight_list)

Found 16 node types.
Found 996 edge types.
Add edge_weights to HeteroData
Conversion complete!


In [99]:
data

HeteroData(
  x=ModuleDict(
  (Gene): Embedding(137, 64)
  (Abundance): Embedding(408, 64)
  (BiologicalProcess): Embedding(446, 64)
  (Activity): Embedding(546, 64)
  (Pathology): Embedding(126, 64)
  (MicroRna): Embedding(46, 64)
  (Protein): Embedding(1301, 64)
  (Rna): Embedding(111, 64)
  (Translocation): Embedding(64, 64)
  (Reaction): Embedding(13, 64)
  (Degradation): Embedding(56, 64)
  (CellSecretion): Embedding(30, 64)
  (CellSurfaceExpression): Embedding(3, 64)
  (Complex): Embedding(373, 64)
  (Composite): Embedding(72, 64)
  (Patient): Embedding(744, 64)
),
  Gene={
    num_nodes=137,
    relevance=[137],
  },
  Abundance={
    num_nodes=408,
    relevance=[408],
  },
  BiologicalProcess={
    num_nodes=446,
    relevance=[446],
  },
  Activity={
    num_nodes=546,
    relevance=[546],
  },
  Pathology={
    num_nodes=126,
    relevance=[126],
  },
  MicroRna={
    num_nodes=46,
    relevance=[46],
  },
  Protein={
    num_nodes=1301,
    relevance=[1301],
  },
  Rna={
  

### Model define

+ ConditioanlMessageLayer:
    + Conditional message passing:
    + patient - protein edge:
      + $ r_1(v) $ = relevance score of non-patient node;
      + $ r_2(u) = \begin{cases} 1 & \text{if }  y_u = 1 \\ 0 & \text{if }  y_u = 0 \end{cases}$, where $u$ is patient ndoe;
      + $ r_3(u,v) = e_{u,v}$, the edge_weight between node $u$ and $v$;
      + combined gating rule:
        + $ g_{u,v} = \sigma(\alpha (r_1(v)*r_3(u,v) - \theta)) * r_2(u) $; 
        + $ g_{u,v} \in [0,1]$
    + all other edges gating:
      + $g_{u,v}$ = edge weight
        + edge_weight = embedding similarity between nodes;(`current implementation`)
        + edge_weight = the average node_relevance of two nodes;(`prefer`)
        + or edge_weight = 1: `not good because gated_edge_weight of edge_patient&protein < 1`
  
+ GNNModel
  + instead of combining gate-layer and GNN layer before the final classifier layer, here the rule-layer should be incorporated in each GNN layer;
  + incorporate gated_edge_weight to GAT layer.

In [143]:
class ConditionalRuleLayer(nn.Module):
    """
    Computes edge-level gates based on: node relevance; patient disease label; edge weight
    """
    def __init__(self):
        super().__init__()
        self.theta = nn.Parameter(torch.zeros(1))
        self.alpha = nn.Parameter(torch.ones(1))

    def forward(
        self,
        dst_relevance,  
        edge_weight,    
        src_is_patient: bool, 
        is_ad_patient: bool        
    ):
        """
        Returns:
            gate: [E] ∈ (0,1)
        """
        base = dst_relevance * edge_weight
        score = torch.sigmoid(self.alpha * (base - self.theta))

        gate = torch.ones_like(score)
        #print('all score: ',score)
        #print('ad_edge score:', score[src_is_patient & is_ad_patient])
        #print('healthy edges score:', 1.0 - score[src_is_patient & ~is_ad_patient])

        # Non-patient-node → AD patient
        gate[src_is_patient & is_ad_patient] = score[src_is_patient & is_ad_patient]

        # Non-patient → Control patient
        gate[src_is_patient & ~is_ad_patient] = 1.0 - score[src_is_patient & ~is_ad_patient]

        return gate

In [ ]:
class ConditionHeteroGAT(torch.nn.Module):
    """Heterogeneous GNN encoder with learnable node embeddings."""
    def __init__(self, data, in_channels, hidden_channels, out_channels, 
                 dropout_rate, heads, aggr='sum', num_layers = 2):
        super().__init__()
        
        self.data = data
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        self.hidden_channels = hidden_channels
        self.out_channels = out_channels

        # GAT backbone
        self.convs = nn.ModuleList()
        self.convs.append(HeteroConv({
            edge_type: GATConv(in_channels, hidden_channels, heads=heads, add_self_loops=False) for edge_type in data.edge_types
        }, aggr=aggr))
        for _ in range(num_layers - 1):
            self.convs.append(HeteroConv({
            edge_type: GATConv(hidden_channels, hidden_channels, heads=1, add_self_loops=False) for edge_type in data.edge_types
        }, aggr=aggr))
        
            # batch normalization?

        # Conditional rules
        self.conditional_rule = ConditionalRuleLayer()

        # final layer
        self.convs.append(HeteroConv({
            edge_type: GATConv(hidden_channels, out_channels, heads=1, add_self_loops=False) for edge_type in data.edge_types
        }, aggr=aggr))



    def forward(self, x_dict, edge_index_dict, edge_weight_dict, relevance_dict, y_labels):
        
        gated_edge_weight_dict = {}

        for edge_type in edge_index_dict:
            src_type, _, dst_type = edge_type
            edge_index = edge_index_dict[edge_type]
            edge_weight = torch.tensor(edge_weight_dict[edge_type])
            #print('edge_weight:',edge_weight)
            
            # Default: no gating
            gated_weight = edge_weight

            if src_type == "Patient" and dst_type != "Patient":
                src_nodes = edge_index[0]
                #print('src_nodes index:',src_nodes)
                dst_nodes = edge_index[1]

                dst_relevance = torch.tensor([relevance_dict[dst_type][dst_node] for dst_node in dst_nodes])
                #print('dst_relevance:', dst_relevance)
                src_labels = torch.tensor([y_labels[src_node] for src_node in src_nodes])
                #print('src_labels:', src_labels)
                
                gate = self.conditional_rule(
                    dst_relevance=dst_relevance,
                    edge_weight=edge_weight,
                    src_is_patient=torch.ones_like(src_labels, dtype=torch.bool),
                    is_ad_patient=(src_labels == 1)
                )

                gated_weight = edge_weight * gate
            
            if src_type != "Patient" and dst_type == "Patient":
                src_nodes = edge_index[0]
                #print('src_nodes index:',src_nodes)
                dst_nodes = edge_index[1]

                src_relevance = torch.tensor([relevance_dict[src_type][src_node] for src_node in src_nodes])
                #print('dst_relevance:', dst_relevance)
                dst_labels = torch.tensor([y_labels[dst_node] for dst_node in dst_nodes])
                #print('src_labels:', src_labels)
                
                #print('dst_is_patient:',torch.ones_like(dst_labels, dtype=torch.bool))
                
                conditional_rule = ConditionalRuleLayer()
                gate = conditional_rule(
                    dst_relevance=src_relevance,
                    edge_weight=edge_weight,
                    src_is_patient=torch.ones_like(dst_labels, dtype=torch.bool),
                    is_ad_patient=(dst_labels == 1)
                )

                gated_weight = edge_weight * gate
            
            gated_edge_weight_dict[edge_type] = gated_weight
        
        # HeteroGAT 
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict, gated_edge_weight_dict)
            x_dict = {key: F.relu(x) for key, x in x_dict.items()}
            x_dict = {key: F.dropout(x, p=self.dropout_rate, training=self.training) for key, x in x_dict.items()}
            

        return x_dict


In [117]:
print('edge_index_dict')
edge_index_dict = {
        edge_type: data[edge_type].edge_index.cpu()
        for edge_type in data.edge_types
    }
for edge_type in edge_index_dict:
    print(edge_type)
    print(edge_index_dict[edge_type])
    break

print('\nedge_weight_dict')
edge_weight_dict = edge_weight_list
print(edge_weight_dict[edge_type])

print('\nrelevance dict')
relevance_dict = node_relevances
print('gene', len(relevance_dict['Gene']))



edge_index_dict
('Gene', 'decreases', 'Protein')
tensor([[  0,   6,  84,  86,  99],
        [269, 269, 269, 269, 261]])

edge_weight_dict
[tensor(-0.1289), tensor(-0.0594), tensor(-0.3320), tensor(-0.3143), tensor(-0.0679)]

relevance dict
gene 137


In [145]:
gated_edge_weight_dict = {}

for edge_type in edge_index_dict:
    src_type, _, dst_type = edge_type
    edge_index = edge_index_dict[edge_type]
    edge_weight = torch.tensor(edge_weight_dict[edge_type])
    #print('edge_weight:',edge_weight)
    
    # Default: no gating
    gated_weight = edge_weight

    if src_type == "Patient" and dst_type != "Patient":
        src_nodes = edge_index[0]
        #print('src_nodes index:',src_nodes)
        dst_nodes = edge_index[1]

        dst_relevance = torch.tensor([relevance_dict[dst_type][dst_node] for dst_node in dst_nodes])
        #print('dst_relevance:', dst_relevance)
        src_labels = torch.tensor([labels[src_node] for src_node in src_nodes])
        #print('src_labels:', src_labels)
        
        conditional_rule = ConditionalRuleLayer()
        gate = conditional_rule(
            dst_relevance=dst_relevance,
            edge_weight=edge_weight,
            src_is_patient=torch.ones_like(src_labels, dtype=torch.bool),
            is_ad_patient=(src_labels == 1)
        )

        gated_weight = edge_weight * gate
        print(edge_type, gate)
    
    if src_type != "Patient" and dst_type == "Patient":
        src_nodes = edge_index[0]
        #print('src_nodes index:',src_nodes)
        dst_nodes = edge_index[1]

        src_relevance = torch.tensor([relevance_dict[src_type][src_node] for src_node in src_nodes])
        #print('dst_relevance:', dst_relevance)
        dst_labels = torch.tensor([labels[dst_node] for dst_node in dst_nodes])
        #print('src_labels:', src_labels)
        
        #print('dst_is_patient:',torch.ones_like(dst_labels, dtype=torch.bool))
        
        conditional_rule = ConditionalRuleLayer()
        gate = conditional_rule(
            dst_relevance=src_relevance,
            edge_weight=edge_weight,
            src_is_patient=torch.ones_like(dst_labels, dtype=torch.bool),
            is_ad_patient=(dst_labels == 1)
        )

        gated_weight = edge_weight * gate
        print(edge_type, gate)
    
    gated_edge_weight_dict[edge_type] = gated_weight

('Protein', 'rev_express', 'Patient') tensor([0.5030, 0.5028, 0.4970,  ..., 0.3203, 0.6718, 0.3191],
       dtype=torch.float64, grad_fn=<IndexPutBackward0>)
('Patient', 'express', 'Protein') tensor([0.5030, 0.4995, 0.5241,  ..., 0.4949, 0.5896, 0.3191],
       dtype=torch.float64, grad_fn=<IndexPutBackward0>)


In [136]:
torch.stack(edge_weight_dict[('Patient',
  'express',
  'Protein')])

tensor([2.6365, 2.4435, 2.5600,  ..., 2.0855, 7.5000, 9.8185],
       dtype=torch.float64)

### (b) Message passing gate at each layer

In [ ]:
class ConditionalGAT(nn.Module):
    """
    A Layer that integrates GAT and Message Gate.
    """

    def __init__(self, in_channels, hidden_channels, out_channels,
                 heads, dropout, negative_slope=0.2):
        self.dropout = dropout
        self.negative_slope = negative_slope
        